# Лабораторная работа 5: SLIM и NNMF

Демонстрация всех этапов: загрузка текстового датасета, обучение SLIM и **NNMF** (Non-Negative Matrix Factorization), оценка **RMSE** и **NDCG** через `sklearn.metrics`, сравнение своих реализаций с библиотечными (`sklearn`).

In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score, root_mean_squared_error

from source.data import load_text_interaction_matrix
from source.evaluation import ranking_ndcg, rating_rmse
from source.nmf import NmfModel
from source.references import LibraryNmfModel, LibrarySlimModel
from source.slim import SlimModel

sklearn metrics: <function root_mean_squared_error at 0x00000203398B4900> <function ndcg_score at 0x0000020339896A20>


## Загрузка текстового датасета

Используем **20 Newsgroups** (`sklearn.datasets`) и строим матрицу документ–термин с `TfidfVectorizer`.

In [13]:
dataset, train, test = load_text_interaction_matrix(
    categories=("sci.med", "sci.space"),
    max_features=300,
    test_size=0.2,
    random_state=42,
)

rows, cols = test.nonzero()
y_test = np.asarray(test[rows, cols]).ravel()

print(f"Документов: {train.shape[0]}")
print(f"Терминов: {train.shape[1]}")
print(f"Train nnz: {train.nnz}, Test nnz: {test.nnz}")
print(f"Категории: {dataset.category_names}")

Документов: 1977
Терминов: 300
Train nnz: 63268, Test nnz: 15816
Категории: ('sci.med', 'sci.space')


## SLIM: обучение, RMSE, сравнение с эталоном

In [14]:
slim = SlimModel(l1_coef=0.5, l2_coef=1.0, max_iter=80)
slim.fit(train)
slim_pred = slim.predict(train)[rows, cols]
slim_full = slim.predict(train)

slim_lib = LibrarySlimModel(l1_coef=0.5, l2_coef=1.0, max_iter=80)
slim_lib.fit(train)
slim_lib_pred = slim_lib.predict(train)[rows, cols]
slim_lib_full = slim_lib.predict(train)

pd.DataFrame(
    {
        "Метрика": ["RMSE", "NDCG"],
        "Своя реализация": [
            rating_rmse(y_test, slim_pred),
            ranking_ndcg(test, slim_full, k=10),
        ],
        "Библиотека (sklearn)": [
            rating_rmse(y_test, slim_lib_pred),
            ranking_ndcg(test, slim_lib_full, k=10),
        ],
    }
)

,Метрика,Своя реализация,Библиотека (sklearn)
0,RMSE,0.154444,0.154444
1,NDCG,0.025571,0.025571


## NNMF: обучение, RMSE, сравнение с библиотекой

**Выбор модели:** для TF-IDF (все веса ≥ 0) выбран **NNMF**, LFM — возможны отрицательные факторы, ALS чаще для collaborative filtering.

In [15]:
# своя NNMF (solver='mu')
nmf = NmfModel(n_components=40, random_state=42)
nmf.fit(train)
nmf_pred = nmf.predict(train)[rows, cols]
nmf_full = nmf.predict(train)

# библиотечная NNMF (sklearn NMF, solver='cd')
nmf_lib = LibraryNmfModel(n_components=40, random_state=42)
nmf_lib.fit(train)
nmf_lib_pred = nmf_lib.predict(train)[rows, cols]
nmf_lib_full = nmf_lib.predict(train)

pd.DataFrame(
    {
        "Метрика": ["RMSE", "NDCG"],
        "Своя реализация": [
            rating_rmse(y_test, nmf_pred),
            ranking_ndcg(test, nmf_full, k=10),
        ],
        "Библиотека (sklearn NMF)": [
            rating_rmse(y_test, nmf_lib_pred),
            ranking_ndcg(test, nmf_lib_full, k=10),
        ],
    }
)

,Метрика,Своя реализация,Библиотека (sklearn NMF)
0,RMSE,0.145153,0.145529
1,NDCG,0.013657,0.012632


## Сводная таблица результатов

In [16]:
summary = pd.DataFrame(
    [
        [
            "SLIM",
            rating_rmse(y_test, slim_pred),
            rating_rmse(y_test, slim_lib_pred),
            ranking_ndcg(test, slim_full, k=10),
            ranking_ndcg(test, slim_lib_full, k=10),
        ],
        [
            "NNMF",
            rating_rmse(y_test, nmf_pred),
            rating_rmse(y_test, nmf_lib_pred),
            ranking_ndcg(test, nmf_full, k=10),
            ranking_ndcg(test, nmf_lib_full, k=10),
        ],
    ],
    columns=[
        "Модель",
        "RMSE (своя)",
        "RMSE (библиотека)",
        "NDCG (своя)",
        "NDCG (библиотека)",
    ],
)
summary

,Модель,RMSE (своя),RMSE (библиотека),NDCG (своя),NDCG (библиотека)
0,SLIM,0.154444,0.154444,0.025571,0.025571
1,NNMF,0.145153,0.145529,0.013657,0.012632


### Выводы

- Для TF-IDF выбран **NNMF**: матрица неотрицательна, латентные темы интерпретируемы.
- Метрики — `sklearn.metrics.root_mean_squared_error` и `sklearn.metrics.ndcg_score`.
- Свои модели: `SlimModel`, `NmfModel`; библиотечные: `LibrarySlimModel`, `LibraryNmfModel`.
- Подробный отчёт — в `README.md`.